<a href="https://colab.research.google.com/github/Akshatha7710/RAG-based-Document-Question-Answering-System-using-LangChain-FAISS-and-LLMs/blob/main/Question%20Answering%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG-Based Document Question Answering System
**Stack:** PyMuPDF · ChromaDB · Sentence-Transformers · Gemini 2.5 Flash


In [1]:
# Cell 1: Install Dependencies
!pip install -q pymupdf chromadb sentence-transformers langchain-text-splitters google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
# Cell 2: Set API Key
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("✅ API Key loaded:", os.environ["GEMINI_API_KEY"][:10], "...")

✅ API Key loaded: AIzaSyDFX_ ...


In [3]:
# Cell 3: Imports & Setup
import fitz                          # PyMuPDF
import chromadb
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time

# Configure Gemini
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-2.5-flash")

# Embedding model (local, no API needed)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Vector database
chroma     = chromadb.Client()
collection = chroma.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

print("✅ All components initialized")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All components initialized


In [4]:
# Cell 4: Core RAG Functions

def ingest_pdf(path: str):
    """Extract text from PDF, chunk it, embed it, store in ChromaDB."""
    doc    = fitz.open(path)
    text   = "\n\n".join(page.get_text() for page in doc)
    doc.close()

    chunks     = splitter.split_text(text)
    embeddings = embedder.encode(chunks).tolist()
    ids        = [f"chunk_{i}" for i in range(len(chunks))]

    collection.upsert(ids=ids, embeddings=embeddings, documents=chunks)
    print(f"✅ Ingested {len(chunks)} chunks from '{path}'")


def ask(question: str) -> str:
    """Retrieve relevant chunks and generate an answer using Gemini."""
    # Embed question and find top 3 matching chunks
    q_emb   = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    context = "\n\n---\n\n".join(results["documents"][0])

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say so clearly.

CONTEXT:
{context}

QUESTION:
{question}"""

    response = model.generate_content(prompt)
    return response.text

print("✅ Functions ready")

✅ Functions ready


In [5]:
# Cell 5: Upload & Ingest PDF
from google.colab import files

uploaded = files.upload()                      # opens file picker
pdf_path = list(uploaded.keys())[0]
ingest_pdf(pdf_path)

Saving ai_basics.pdf to ai_basics.pdf
✅ Ingested 13 chunks from 'ai_basics.pdf'


In [6]:
# Cell 6: Ask Questions
questions = [
    "What are the three types of machine learning?",
    "How is AI used in healthcare?",
    "What are the ethical concerns around AI?",
    "What is the difference between deep learning and machine learning?"
]

for q in questions:
    print("\n" + "="*60)
    print(f"Q: {q}")
    print(ask(q))
    time.sleep(30)   # 30 seconds between questions — stays within 5/min limit


Q: What are the three types of machine learning?
The three types of machine learning are:
- Supervised Learning
- Unsupervised Learning
- Reinforcement Learning

Q: How is AI used in healthcare?
AI is used in healthcare to:
*   Identify patients at risk of deterioration through predictive analytics, enabling earlier intervention.
*   Power virtual health assistants that help patients manage chronic conditions and answer medical questions around the clock.

Q: What are the ethical concerns around AI?
The ethical concerns around AI include:
*   Algorithmic bias, where AI systems can perpetuate or amplify existing societal biases.
*   Privacy concerns around data collection and surveillance.
*   The impact of automation on employment.
*   The challenge of making AI systems explainable and transparent.

Q: What is the difference between deep learning and machine learning?
Deep learning is a subset of machine learning.

Deep learning uses neural networks with many layers to model complex p

In [7]:
# Cell 7: Ask Your Own Question
my_question = "Summarize the document in 3 bullet points"  # ← change this

print(f"Q: {my_question}")
print(ask(my_question))

Q: Summarize the document in 3 bullet points


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2481.78ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4935.97ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1874.58ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1549.02ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2760.53ms


*   Natural Language Processing (NLP) is a branch of AI that enables computers to understand and generate human language, with Large Language Models (LLMs) like GPT and Claude performing tasks such as writing, coding, and answering questions.
*   The future of AI involves the long-term goal of Artificial General Intelligence (AGI), increasing integration into everyday tools, and the emergence of multimodal AI systems.
*   Responsible development that addresses bias, privacy, and transparency is essential for AI, which is transforming sectors like healthcare and finance, with governments working on ethical frameworks.
